# Tier 1 merged analysis (four merged conditions)

Reads `eval_results/tier1_acl_results.csv` and merges `hp`, `hp+sg`, `hp_wiener` into
one `high_pass` condition, analogous to the Tier 2 merged analysis. Outputs go to
`eval_results/derived/`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

BASE = Path(r"C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results")
DERIVED = BASE / "derived"
DERIVED.mkdir(parents=True, exist_ok=True)
RAW_PATH = BASE / "tier1_acl_results.csv"

CONDITIONS = ["arm_a", "high_pass", "hp_ffmpeg", "hp_wiener_oa"]
NONBASE = ["high_pass", "hp_ffmpeg", "hp_wiener_oa"]
KEYS = ["talk_id", "noise_type", "snr_db", "noise_seed"]
B = 5000
SEED = 0

In [2]:
# Report expected columns not found in the raw CSV.
raw_expected = ["tier","talk_id","condition","noise_type","snr_db","noise_seed","wer","cer",
                "rtf","seconds","audio_duration_s","kter","term_presence_recall",
                "signal_snr_improvement_db","language","duration_bucket","nshr"]
raw = pd.read_csv(RAW_PATH)
print("Raw columns:", raw.columns.tolist())
print("Raw missing expected:", [c for c in raw_expected if c not in raw.columns.tolist()])

Raw columns: ['tier', 'split', 'talk_id', 'condition', 'noise_type', 'snr_db', 'noise_seed', 'wer', 'cer', 'kter', 'term_presence_recall', 'terms_present', 'signal_snr_improvement_db', 'rtf', 'audio_duration_s', 'seconds', 'language']
Raw missing expected: ['duration_bucket', 'nshr']


In [3]:
# Prepare data (mirror tier1_analysis.ipynb conventions)
df = raw.copy()
df["snr_db"] = df["snr_db"].fillna("clean").map(lambda v: "clean" if v == "clean" else str(int(float(v))))
df["noise_seed"] = df["noise_seed"].fillna("clean").astype(str)
df["condition"] = df["condition"].astype(str)

# Build four merged conditions: average hp / hp+sg / hp_wiener into high_pass.
high = ["hp", "hp+sg", "hp_wiener"]
hp = df[df.condition.isin(high)].groupby(KEYS, dropna=False)[["wer","cer","rtf","seconds","audio_duration_s"]].mean().reset_index()
hp["condition"] = "high_pass"
other = df[~df.condition.isin(high)].copy()
merged = pd.concat([other, hp], ignore_index=True)
merged["wer_pct"] = 100 * merged["wer"]
merged["cer_pct"] = 100 * merged["cer"]
merged.to_csv(DERIVED / "tier1_merged_conditions.csv", index=False)
print("merged rows:", len(merged), "scenarios:", len(merged) // 4, "talks:", merged.talk_id.nunique())
print("conditions:", merged.condition.value_counts().to_dict())

merged rows: 260 scenarios: 65 talks: 5
conditions: {'arm_a': 65, 'hp_wiener_oa': 65, 'hp_ffmpeg': 65, 'high_pass': 65}


C:\Users\Leon\AppData\Local\Temp\ipykernel_25632\973000307.py:3: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  df["snr_db"] = df["snr_db"].fillna("clean").map(lambda v: "clean" if v == "clean" else str(int(float(v))))
C:\Users\Leon\AppData\Local\Temp\ipykernel_25632\973000307.py:4: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  df["noise_seed"] = df["noise_seed"].fillna("clean").astype(str)


## 1. Accuracy table (mean WER/CER %, with n)

In [4]:
acc_rows = []
def acc_for(df_in, scope):
    for c in CONDITIONS:
        g = df_in[df_in.condition == c]
        acc_rows.append({"scope": scope, "condition": c, "n": len(g),
                        "wer_mean_pct": round(100*g["wer"].mean(),4),
                        "cer_mean_pct": round(100*g["cer"].mean(),4)})

acc_for(merged, "overall")
for s in sorted(merged.snr_db.unique()):
    acc_for(merged[merged.snr_db == s], str(s))
acc_df = pd.DataFrame(acc_rows)
print(acc_df.to_string(index=False))
acc_df.to_csv(DERIVED / "tier1_merged_accuracy.csv", index=False)

  scope    condition  n  wer_mean_pct  cer_mean_pct
overall        arm_a 65       19.3257        9.6096
overall    high_pass 65       19.2063        9.5090
overall    hp_ffmpeg 65       20.0540       10.0231
overall hp_wiener_oa 65       19.3505        9.5682
      0        arm_a 15       23.0921       12.2433
      0    high_pass 15       22.4401       11.7393
      0    hp_ffmpeg 15       23.6248       12.5249
      0 hp_wiener_oa 15       22.3649       11.6910
     10        arm_a 15       18.4796        8.8953
     10    high_pass 15       18.4103        8.9036
     10    hp_ffmpeg 15       19.2494        9.3972
     10 hp_wiener_oa 15       18.5829        8.9937
     20        arm_a 15       16.8720        8.0328
     20    high_pass 15       17.0740        8.0426
     20    hp_ffmpeg 15       17.8579        8.5932
     20 hp_wiener_oa 15       17.6344        8.3122
      5        arm_a 15       19.6478        9.8096
      5    high_pass 15       19.7144        9.8893
      5    h

## 2. Paired deltas, cluster bootstrap over talk_id, Wilcoxon/Holm, Cohen's dz, win/loss/tie

In [5]:
base = merged[merged.condition == "arm_a"].set_index(KEYS)
delta_rows = []
for c in NONBASE:
    cdf = merged[merged.condition == c].set_index(KEYS)
    common = base.index.intersection(cdf.index)
    for key in common:
        delta_rows.append({
            "talk_id": key[0], "noise_type": key[1], "snr_db": key[2], "noise_seed": key[3],
            "condition": c,
            "delta_wer_pp": 100*(base.loc[key,"wer"] - cdf.loc[key,"wer"]),
            "delta_cer_pp": 100*(base.loc[key,"cer"] - cdf.loc[key,"cer"]),
        })
delta_df = pd.DataFrame(delta_rows)
print("paired deltas:", delta_df.shape)
delta_df.to_csv(DERIVED / "tier1_merged_paired_deltas.csv", index=False)

def cluster_ci(values_df, value_col, cluster_col="talk_id", B=B, seed=SEED):
    clusters = values_df[cluster_col].unique()
    vals = values_df[value_col].to_numpy(float)
    cl = values_df[cluster_col].to_numpy()
    idx_by_cluster = {c: np.where(cl == c)[0] for c in clusters}
    rng = np.random.default_rng(seed)
    boot = np.empty(B)
    for b in range(B):
        chosen = rng.choice(clusters, size=len(clusters), replace=True)
        rows = np.concatenate([idx_by_cluster[c] for c in chosen])
        boot[b] = vals[rows].mean()
    return float(vals.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

ci_rows = []
for scope in ["overall"] + sorted(merged.snr_db.unique()):
    dscope = delta_df if scope == "overall" else delta_df[delta_df.snr_db == scope]
    for c in NONBASE:
        for metric, col in [("WER","delta_wer_pp"),("CER","delta_cer_pp")]:
            sub = dscope[dscope.condition == c]
            mean, lo, hi = cluster_ci(sub, col)
            ci_rows.append({"scope": scope, "condition": c, "metric": metric,
                            "n": len(sub), "mean_delta_pp": round(mean,4),
                            "ci_lo_pp": round(lo,4), "ci_hi_pp": round(hi,4)})
ci_df = pd.DataFrame(ci_rows)
print(ci_df.to_string(index=False))
ci_df.to_csv(DERIVED / "tier1_merged_bootstrap_ci.csv", index=False)

paired deltas: (195, 7)
  scope    condition metric  n  mean_delta_pp  ci_lo_pp  ci_hi_pp
overall    high_pass    WER 65         0.1194   -0.1473    0.4905
overall    high_pass    CER 65         0.1006   -0.0583    0.2594
overall    hp_ffmpeg    WER 65        -0.7283   -0.8623   -0.5958
overall    hp_ffmpeg    CER 65        -0.4135   -0.6491   -0.2592
overall hp_wiener_oa    WER 65        -0.0248   -0.2958    0.3312
overall hp_wiener_oa    CER 65         0.0414   -0.0679    0.1508
      0    high_pass    WER 15         0.6520   -0.3258    1.7809
      0    high_pass    CER 15         0.5040   -0.1253    1.2593
      0    hp_ffmpeg    WER 15        -0.5327   -1.0393    0.0822
      0    hp_ffmpeg    CER 15        -0.2816   -0.8257    0.2360
      0 hp_wiener_oa    WER 15         0.7272   -0.2910    1.9721
      0 hp_wiener_oa    CER 15         0.5523   -0.0764    1.2912
     10    high_pass    WER 15         0.0693   -0.3950    0.5766
     10    high_pass    CER 15        -0.0084   -0.1

In [6]:
def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    for rank, i in enumerate(order):
        adj[i] = p[i] * (m - rank)
    adj_sorted = np.minimum(1.0, np.maximum.accumulate(adj[order]))
    out = np.empty(m)
    out[order] = adj_sorted
    return np.minimum(1.0, out)

sig_rows = []
for metric, col in [("WER","delta_wer_pp"),("CER","delta_cer_pp")]:
    pvals = []
    for c in NONBASE:
        x = delta_df[delta_df.condition == c][col].to_numpy(float)
        w = stats.wilcoxon(x, alternative="two-sided", method="auto")
        dz = float(x.mean()/x.std(ddof=1)) if x.std(ddof=1)>0 else float("nan")
        pvals.append(float(w.pvalue))
        sig_rows.append({"metric": metric, "condition": c, "n": len(x),
                        "mean_delta_pp": round(float(x.mean()),4), "cohen_dz": round(dz,4),
                        "wilcoxon_statistic": float(w.statistic), "p_value": w.pvalue,
                        "win": int((x>0).sum()), "loss": int((x<0).sum()), "tie": int((x==0).sum())})
sig_df = pd.DataFrame(sig_rows)
for metric in ["WER","CER"]:
    mask = sig_df.metric == metric
    sig_df.loc[mask, "p_holm"] = holm_adjust(sig_df.loc[mask,"p_value"].to_numpy())
sig_df["p_value"] = sig_df["p_value"].round(6)
sig_df["p_holm"] = sig_df["p_holm"].round(6)
print(sig_df.to_string(index=False))
sig_df.to_csv(DERIVED / "tier1_merged_significance.csv", index=False)

metric    condition  n  mean_delta_pp  cohen_dz  wilcoxon_statistic  p_value  win  loss  tie   p_holm
   WER    high_pass 65         0.1194    0.1154               997.0 0.939969   28    35    2 0.939969
   WER    hp_ffmpeg 65        -0.7283   -0.6916               276.0 0.000000    9    55    1 0.000001
   WER hp_wiener_oa 65        -0.0248   -0.0203               873.5 0.265500   28    36    1 0.531000
   CER    high_pass 65         0.1006    0.1573              1021.5 0.738921   32    33    0 1.000000
   CER    hp_ffmpeg 65        -0.4135   -0.5291               352.0 0.000004   10    54    1 0.000013
   CER hp_wiener_oa 65         0.0414    0.0558               982.5 0.700581   28    36    1 1.000000


## 3. Timing table from raw CSV

In [7]:
r = raw.copy()
r["snr_db"] = r["snr_db"].fillna("clean").map(lambda v: "clean" if v == "clean" else str(int(float(v))))
r["noise_seed"] = r["noise_seed"].fillna("clean").astype(str)
r["merged_condition"] = r["condition"].map({"arm_a":"arm_a","hp":"high_pass","hp+sg":"high_pass",
                                          "hp_wiener":"high_pass","hp_ffmpeg":"hp_ffmpeg",
                                          "hp_wiener_oa":"hp_wiener_oa"})

timing_rows = []
for c in CONDITIONS:
    if c == "high_pass":
        sub = r[r.condition.isin(["hp","hp+sg","hp_wiener"])]
    else:
        sub = r[r.condition == c]
    timing_rows.append({"condition": c, "n": len(sub),
                       "rtf_mean": round(float(sub.rtf.mean()),4),
                       "rtf_sd": round(float(sub.rtf.std(ddof=1)),4),
                       "seconds_mean": round(float(sub.seconds.mean()),4),
                       "seconds_sd": round(float(sub.seconds.std(ddof=1)),4)})
timing_df = pd.DataFrame(timing_rows)

# paired overhead vs arm_a per scenario (high_pass = average of hp/hp+sg/hp_wiener)
keys = KEYS
arm_t = r[r.condition == "arm_a"].set_index(keys)
hp_t = r[r.condition.isin(["hp","hp+sg","hp_wiener"])].groupby(keys)[["rtf","seconds"]].mean()

over = {}
over["high_pass"] = (hp_t["rtf"] - arm_t["rtf"], hp_t["seconds"] - arm_t["seconds"])
for c in ["hp_ffmpeg","hp_wiener_oa"]:
    ct = r[r.condition == c].set_index(keys)
    over[c] = (ct["rtf"] - arm_t["rtf"], ct["seconds"] - arm_t["seconds"])

over_rows = []
for c in NONBASE:
    dr, ds = over[c]
    tmp = pd.DataFrame({"talk_id": [k[0] for k in dr.index], "delta_rtf": dr.to_numpy(), "delta_seconds": ds.to_numpy()})
    mr, lorr, hirr = cluster_ci(tmp, "delta_rtf")
    ms, lors, hiss = cluster_ci(tmp, "delta_seconds")
    over_rows.append({"condition": c,
                     "overhead_rtf_mean": round(mr,4), "overhead_rtf_ci_lo": round(lorr,4), "overhead_rtf_ci_hi": round(hirr,4),
                     "overhead_seconds_mean": round(ms,4), "overhead_seconds_ci_lo": round(lors,4), "overhead_seconds_ci_hi": round(hiss,4)})
over_df = pd.DataFrame(over_rows)
timing_df = timing_df.merge(over_df, on="condition", how="left")
timing_df.loc[timing_df.condition=="arm_a", ["overhead_rtf_mean","overhead_rtf_ci_lo","overhead_rtf_ci_hi",
                                            "overhead_seconds_mean","overhead_seconds_ci_lo","overhead_seconds_ci_hi"]] = 0.0
print(timing_df.to_string(index=False))
timing_df.to_csv(DERIVED / "tier1_merged_timing.csv", index=False)

C:\Users\Leon\AppData\Local\Temp\ipykernel_25632\868135117.py:2: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  r["snr_db"] = r["snr_db"].fillna("clean").map(lambda v: "clean" if v == "clean" else str(int(float(v))))
C:\Users\Leon\AppData\Local\Temp\ipykernel_25632\868135117.py:3: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  r["noise_seed"] = r["noise_seed"].fillna("clean").astype(str)


   condition   n  rtf_mean  rtf_sd  seconds_mean  seconds_sd  overhead_rtf_mean  overhead_rtf_ci_lo  overhead_rtf_ci_hi  overhead_seconds_mean  overhead_seconds_ci_lo  overhead_seconds_ci_hi
       arm_a  65    0.0720  0.0048       49.2687      6.7061             0.0000              0.0000              0.0000                 0.0000                  0.0000                  0.0000
   high_pass 195    0.0840  0.0048       57.4243      7.2661             0.0120              0.0113              0.0124                 8.1556                  7.4776                  8.8336
   hp_ffmpeg  65    0.0797  0.0047       54.4845      7.0703             0.0077              0.0071              0.0083                 5.2158                  4.6430                  5.7411
hp_wiener_oa  65    0.0822  0.0050       56.1880      7.2894             0.0102              0.0096              0.0109                 6.9193                  6.2487                  7.5223


## 4. Figures (four merged conditions only)

In [8]:
m4 = merged[merged.condition.isin(CONDITIONS)].copy()
means = m4.groupby("condition")["wer"].mean().reindex(CONDITIONS)
fig, ax = plt.subplots(figsize=(7,5))
ax.bar(means.index, 100*means.values, color="#4c78a8")
ax.set_title("Tier 1 mean WER by merged condition")
ax.set_ylabel("WER (%)")
ax.set_xticks(range(len(means.index)))
ax.set_xticklabels(means.index, rotation=30, ha="right")
fig.tight_layout()
fig.savefig(DERIVED / "tier1_merged_wer_by_condition.png", dpi=150)
plt.close(fig)

snr_means = m4.groupby(["snr_db","condition"])["wer"].mean().unstack("condition").reindex(columns=CONDITIONS)
fig, ax = plt.subplots(figsize=(8,5))
snr_means.plot.bar(ax=ax)
ax.set_title("Tier 1 WER by input SNR and merged condition")
ax.set_ylabel("WER (%)")
fig.tight_layout()
fig.savefig(DERIVED / "tier1_merged_wer_by_snr.png", dpi=150)
fig.savefig(DERIVED / "tier1_merged_wer_by_bucket.png", dpi=150)
plt.close(fig)
print("Saved figures to", DERIVED)

Saved figures to C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived


## 5. Summary

In [9]:
print("Saved tables:")
for f in ["tier1_merged_accuracy.csv","tier1_merged_paired_deltas.csv","tier1_merged_bootstrap_ci.csv",
          "tier1_merged_significance.csv","tier1_merged_timing.csv","tier1_merged_conditions.csv"]:
    print(" -", DERIVED / f)
print("Saved figures:")
for f in ["tier1_merged_wer_by_condition.png","tier1_merged_wer_by_snr.png","tier1_merged_wer_by_bucket.png"]:
    print(" -", DERIVED / f)

print()
for _, row in acc_df[acc_df.scope=="overall"].iterrows():
    print(f"  {row.condition}: WER {row.wer_mean_pct:.2f}%, CER {row.cer_mean_pct:.2f}% (n={row.n})")
print()
for _, row in ci_df[(ci_df.scope=='overall')&(ci_df.metric=='WER')].iterrows():
    print(f"  {row.condition} vs arm_a delta WER = {row.mean_delta_pp:+.3f} pp (95% CI {row.ci_lo_pp:+.3f}, {row.ci_hi_pp:+.3f})")
print()
print("Missing expected columns reported above.")

Saved tables:
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_accuracy.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_paired_deltas.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_bootstrap_ci.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_significance.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_timing.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_conditions.csv
Saved figures:
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_wer_by_condition.png
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_wer_by_snr.png
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results\derived\tier1_merged_wer_by_bucket.png

  arm_a: WER 19.33%, CER 9.61% (n

In [10]:
# Table 2: Tier 1 mean WER / CER (%) by condition and input SNR
order = ["clean", "20", "10", "5", "0"]
labels = {"clean": "Clean", "20": "20 dB", "10": "10 dB", "5": "5 dB", "0": "0 dB"}

def wer_cer_cells(sub):
    means = sub.groupby("condition")[["wer", "cer"]].mean().reindex(CONDITIONS) * 100
    return {c: f"{means.loc[c, 'wer']:.2f} / {means.loc[c, 'cer']:.2f}" for c in CONDITIONS}

rows = []
for s in order:
    sub = merged[merged.snr_db == s]
    rows.append({"Input SNR": labels[s], "n": int((sub.condition == "arm_a").sum()), **wer_cer_cells(sub)})
rows.append({"Input SNR": "All", "n": int((merged.condition == "arm_a").sum()), **wer_cer_cells(merged)})

table2 = pd.DataFrame(rows)
print(table2.to_string(index=False))
table2.to_csv(DERIVED / "tier1_table2_wer_cer_by_snr.csv", index=False)

Input SNR  n         arm_a     high_pass     hp_ffmpeg  hp_wiener_oa
    Clean  5  16.96 / 7.98  16.77 / 7.89  17.54 / 8.19  16.85 / 8.14
    20 dB 15  16.87 / 8.03  17.07 / 8.04  17.86 / 8.59  17.63 / 8.31
    10 dB 15  18.48 / 8.90  18.41 / 8.90  19.25 / 9.40  18.58 / 8.99
     5 dB 15  19.65 / 9.81  19.71 / 9.89 20.32 / 10.19  19.65 / 9.75
     0 dB 15 23.09 / 12.24 22.44 / 11.74 23.62 / 12.52 22.36 / 11.69
      All 65  19.33 / 9.61  19.21 / 9.51 20.05 / 10.02  19.35 / 9.57
